# Milestone 4C: SHAP-Based Adverse Action Reasons

Under ECOA Reg B, lenders must provide specific principal reasons for any credit denial. This section builds the technical pipeline for generating those reasons from the production model using SHAP.

For each declined applicant:
1. Compute SHAP values (per-feature attributions to the predicted PD)
2. Extract the top features that pushed PD above the decline threshold
3. Translate feature names + values into plain-English reasons

The output is a demonstration of the pipeline plus 3-4 worked examples. In production, this would be the input to templated notice generation for actual applicants.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

import lightgbm as lgb
import shap
from sklearn.isotonic import IsotonicRegression

from src.data import load_joined, basic_clean
from src.splits import get_splits
from src.features import prepare_features, get_feature_columns

# Load data
df = load_joined()
df = basic_clean(df)
train_idx, val_idx, test_idx = get_splits(df)

# Prepare features (with EXT_SOURCE_1 — the primary production model)
df_features = prepare_features(df, include_ext_source_1=True)
train = df_features.loc[train_idx]
val = df_features.loc[val_idx]

feature_cols = get_feature_columns(include_ext_source_1=True)
categorical_cols = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE",
]

X_train, y_train = train[feature_cols], train["TARGET"]
X_val, y_val = val[feature_cols], val["TARGET"]

# Convert categoricals
X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()
for col in categorical_cols:
    X_train_lgb[col] = X_train_lgb[col].astype("category")
    X_val_lgb[col] = X_val_lgb[col].astype("category")

# Load threshold from 4A
with open(Path("../data/processed/threshold_metadata.pkl"), "rb") as f:
    threshold_meta = pickle.load(f)
tau = threshold_meta["chosen_threshold"]
print(f"Using primary threshold: τ = {tau:.4f}")

In [ ]:
# Same monotonic constraints as Milestone 3B
constraint_directions = {
    "EXT_SOURCE_1": -1, "EXT_SOURCE_2": -1, "EXT_SOURCE_3": -1,
    "bureau_overdue_max": +1, "employment_years": -1, "age_years": -1,
    "payment_to_income": +1, "loan_to_income": +1,
}
monotone_constraints = [constraint_directions.get(col, 0) for col in feature_cols]

lgb_params = {
    "objective": "binary", "metric": "auc",
    "learning_rate": 0.05, "num_leaves": 63, "max_depth": -1,
    "min_data_in_leaf": 100, "feature_fraction": 0.8,
    "bagging_fraction": 0.8, "bagging_freq": 5,
    "random_state": 42, "verbose": -1,
    "monotone_constraints": monotone_constraints,
    "monotone_constraints_method": "intermediate",
}

lgb_train_ds = lgb.Dataset(X_train_lgb, label=y_train, categorical_feature=categorical_cols)
lgb_val_ds = lgb.Dataset(X_val_lgb, label=y_val, categorical_feature=categorical_cols, reference=lgb_train_ds)

lgb_model = lgb.train(
    lgb_params, lgb_train_ds, num_boost_round=2000,
    valid_sets=[lgb_train_ds, lgb_val_ds], valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=0)],
)
print(f"Model retrained. Best iteration: {lgb_model.best_iteration}")

In [ ]:
# Compute SHAP values on a sample to keep runtime manageable (full val set works too)
sample_size = 5000
np.random.seed(42)
sample_idx = np.random.choice(len(X_val_lgb), size=sample_size, replace=False)
X_val_sample = X_val_lgb.iloc[sample_idx].copy()
y_val_sample = y_val.iloc[sample_idx].copy()

explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_val_sample)

print(f"SHAP values computed for {sample_size:,} validation applicants")
print(f"SHAP array shape: {shap_values.shape}")
# One row per applicant, one column per feature

In [ ]:
# Raw predictions
raw_preds = lgb_model.predict(X_val_sample, num_iteration=lgb_model.best_iteration)

# Fit isotonic calibrator on full val set (same as Milestone 3)
val_raw_all = lgb_model.predict(X_val_lgb, num_iteration=lgb_model.best_iteration)
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(val_raw_all, y_val.values)
calibrated_preds = iso.predict(raw_preds)

# Identify declined applicants (calibrated PD > threshold)
declined_mask = calibrated_preds > tau
n_declined = declined_mask.sum()
print(f"Of {sample_size:,} sample applicants:")
print(f"  Declined (PD > {tau:.4f}): {n_declined:,} ({declined_mask.mean():.1%})")
print(f"  Approved: {sample_size - n_declined:,}")

In [ ]:
# Direction of feature value that increases predicted default risk
# "lower" = low values push PD up; "higher" = high values push PD up
FEATURE_DIRECTIONS = {
    # External credit scores
    "EXT_SOURCE_1": "lower",
    "EXT_SOURCE_2": "lower",
    "EXT_SOURCE_3": "lower",
    # Bureau history
    "bureau_overdue_max": "higher",
    "bureau_credit_debt_mean": "higher",
    # Employment and age
    "employment_years": "lower",
    "age_years": "lower",
    # Loan burden ratios
    "payment_to_income": "higher",
    "loan_to_income": "higher",
    "financing_premium": "higher",
    # Income
    "AMT_INCOME_TOTAL": "lower",
    "income_log": "lower",
    # Loan size
    "AMT_CREDIT": "higher",
    "AMT_ANNUITY": "higher",
}

# Human-readable descriptions
FEATURE_DESCRIPTIONS = {
    "EXT_SOURCE_1": "external credit score (source 1)",
    "EXT_SOURCE_2": "external credit score (source 2)",
    "EXT_SOURCE_3": "external credit score (source 3)",
    "bureau_overdue_max": "maximum past-due amount on prior credits",
    "bureau_credit_debt_mean": "average outstanding debt on prior credits",
    "bureau_credit_sum_mean": "average size of prior credits",
    "bureau_credit_sum_max": "largest prior credit",
    "bureau_days_credit_mean": "average age of prior credit accounts",
    "bureau_count_recent": "number of prior credits opened recently",
    "bureau_active_count": "number of active prior credits",
    "employment_years": "length of current employment",
    "age_years": "age of applicant",
    "payment_to_income": "loan payment as fraction of income",
    "loan_to_income": "loan amount as multiple of income",
    "financing_premium": "loan amount vs. price of goods financed",
    "AMT_INCOME_TOTAL": "annual income",
    "income_log": "annual income",
    "AMT_CREDIT": "requested loan amount",
    "AMT_ANNUITY": "annual loan payment",
    "OCCUPATION_TYPE": "occupation category",
    "NAME_EDUCATION_TYPE": "education level",
    "REGION_RATING_CLIENT_W_CITY": "regional risk rating",
    "AMT_REQ_CREDIT_BUREAU_YEAR": "number of credit inquiries in past year",
    "DAYS_ID_PUBLISH": "recency of ID document",
    "DAYS_REGISTRATION": "recency of address registration",
}


def describe_reason(feature_name, feature_value, shap_value):
    """
    Turn a feature+SHAP attribution into a plain-English reason for denial.
    Direction of value tells the applicant what specifically is a problem.
    """
    desc = FEATURE_DESCRIPTIONS.get(feature_name, feature_name)
    direction = FEATURE_DIRECTIONS.get(feature_name, None)

    # Missing values get a specific mention
    if pd.isna(feature_value):
        return f"{desc}: no information available (missing data)"

    if direction == "lower":
        # Low value pushed PD up → tell applicant the value is low
        return f"{desc} is low"
    elif direction == "higher":
        # High value pushed PD up → tell applicant the value is high
        return f"{desc} is high"
    else:
        return f"{desc}: contributed to declination"

In [ ]:
def get_top_reasons(shap_row, feature_row, top_k=4):
    """Return top-K positive SHAP contributions with plain-English descriptions."""
    reasons = []
    for i, (feat, shap_val) in enumerate(zip(X_val_sample.columns, shap_row)):
        if shap_val > 0:  # positive contribution = increased PD
            reasons.append({
                "feature": feat,
                "value": feature_row[feat],
                "shap": shap_val,
                "description": describe_reason(feat, feature_row[feat], shap_val),
            })
    reasons.sort(key=lambda x: x["shap"], reverse=True)
    return reasons[:top_k]


def format_notice(sample_idx, top_k=4):
    """Generate a full adverse action notice for one applicant."""
    shap_row = shap_values[sample_idx]
    feature_row = X_val_sample.iloc[sample_idx]
    predicted_pd = calibrated_preds[sample_idx]
    actual = y_val_sample.iloc[sample_idx]

    reasons = get_top_reasons(shap_row, feature_row, top_k)

    notice = f"--- Applicant {sample_idx} ---\n"
    notice += f"Calibrated PD: {predicted_pd:.4f}  |  Threshold: {tau:.4f}  |  Actual outcome: {'default' if actual == 1 else 'paid'}\n"
    notice += f"Decision: {'DECLINED' if predicted_pd > tau else 'APPROVED'}\n"
    if predicted_pd > tau:
        notice += "\nPrincipal reasons for declination:\n"
        for r in reasons:
            val_str = f"{r['value']:.2f}" if isinstance(r['value'], (int, float)) and not pd.isna(r['value']) else str(r['value'])
            notice += f"  - {r['description']} (feature value: {val_str}, SHAP: {r['shap']:+.4f})\n"
    return notice


# Show 4 declined examples with varied PD levels
declined_indices = np.where(declined_mask)[0]
declined_pds = calibrated_preds[declined_mask]

# Pick examples at different PD quartiles among the declined
pd_percentiles = np.percentile(declined_pds, [25, 50, 75, 95])
example_indices = []
seen = set()
for target_pd in pd_percentiles:
    closest = declined_indices[np.argmin(np.abs(declined_pds - target_pd))]
    if closest not in seen:
        example_indices.append(closest)
        seen.add(closest)

for idx in example_indices:
    print(format_notice(idx, top_k=4))
    print()

In [ ]:
# Global SHAP summary: features ranked by average absolute impact
# Each dot is one applicant; color = feature value, position = SHAP value
shap.summary_plot(shap_values, X_val_sample, feature_names=feature_cols, max_display=15, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# Show waterfall for the median-PD declined applicant (example_indices[1])
idx = example_indices[1]  # median-PD declined applicant

shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value,
    shap_values[idx],
    feature_names=feature_cols,
    max_display=10,
)
plt.tight_layout()
plt.show()

print(f"\nApplicant {idx}:")
print(f"  Baseline (mean prediction): {explainer.expected_value:.4f}")
print(f"  This applicant's prediction: {raw_preds[idx]:.4f}")
print(f"  Calibrated PD: {calibrated_preds[idx]:.4f}")

In [ ]:
# Save example notices for the blog post
import json

example_notices = []
for idx in example_indices:
    shap_row = shap_values[idx]
    feature_row = X_val_sample.iloc[idx]
    reasons = get_top_reasons(shap_row, feature_row, top_k=4)
    example_notices.append({
        "applicant_id": int(idx),
        "calibrated_pd": float(calibrated_preds[idx]),
        "threshold": float(tau),
        "actual_outcome": "default" if y_val_sample.iloc[idx] == 1 else "paid",
        "reasons": [
            {
                "feature": r["feature"],
                "value": float(r["value"]) if pd.notna(r["value"]) else None,
                "shap": float(r["shap"]),
                "description": r["description"],
            }
            for r in reasons
        ],
    })

with open(Path("../data/processed/adverse_action_examples.json"), "w") as f:
    json.dump(example_notices, f, indent=2)

print(f"Saved {len(example_notices)} adverse action examples.")

**Section 4C takeaways:**

**The pipeline:**
1. Retrained the production LightGBM (constrained + calibrated) -- same model architecture as Milestone 3
2. TreeSHAP computed per-applicant feature attributions for a 5,000-applicant validation sample (declined subset: 590 applicants, 11.8%)
3. For each declined applicant, the top-K positive SHAP contributions form the principal reasons for denial
4. A feature-to-plain-English translator converts SHAP attributions into notice-ready reasons: feature descriptions from a lookup table, direction ("low"/"high"/"missing") derived from the feature's expected relationship with default risk, and the applicant's actual feature value included for specificity

**Worked examples:** Three declined applicants at different PD levels:
- PD 0.19 (25th percentile of declined): low external scores across all three sources plus short employment tenure
- PD 0.27 (50th percentile): very low EXT_SOURCE_2 (0.03, near the floor), high financing premium, short employment, missing EXT_SOURCE_1
- PD 0.40 (75th percentile): low EXT_SOURCE_3 and 2, high financing premium, high payment-to-income (87% of annual income)

Each notice contains 3-4 specific, applicant-level reasons tied to actual feature values. The reasons are ranked by SHAP contribution and would form the "specific principal reasons for denial" required under ECOA Reg B.

**What this demonstrates:**

- **The production model is compatible with Reg B.** SHAP values provide the specific principal reasons required by regulation, and the plain-English translator converts them into notice-ready language.
- **TreeSHAP scales.** Computing SHAP values for 5,000 applicants took seconds; production systems can generate these on-demand for every declined applicant without batch delay.
- **Missing values are legitimate reasons.** For thin-file applicants without EXT_SOURCE_1, the notice honestly conveys "no information available (missing data)." This is more truthful than a scorecard would be, and matches how missingness itself was a risk signal in EDA.
- **The waterfall plot reveals what the notice hides.** For Applicant 56, EXT_SOURCE_3 actually reduced predicted risk (SHAP -0.47) but wasn't enough to offset the other reasons. In a compliance context, only the increasing factors are reported -- the notice is for the applicant, not for the modeler. This distinction matters.

**Broader observation on individual vs portfolio fairness:**

Two of the three example declined applicants actually would have paid. This isn't a bug -- at a 87.7% approval threshold, most declined applicants would repay if approved. This is the cost of aggressive risk management. Individually, most FPs feel unfair; in aggregate, the threshold minimizes expected cost. Adverse action notices are legally required precisely because the individual applicant has a legitimate grievance the model cannot answer: "you're statistically riskier than we can tolerate, but you specifically might have been fine."

**Limitations:**
- The plain-English translator here is a stub. Production versions handle edge cases more robustly (numeric formatting for very small/large values, better handling of categorical features, mapping to standardized reason codes for regulatory reporting), and undergo compliance review before deployment.
- SHAP values are computed on raw (uncalibrated) predictions. Isotonic calibration is applied after -- the ranking of features is preserved but the absolute values shift.
- Reg B has additional requirements beyond providing reasons: delivery timing (within 30 days of decision), format (written), applicant rights information, and specific language requirements. A production notice would meet all of these; this analysis focuses only on the reason-generation logic.